In [ ]:
%load_ext autoreload
%autoreload 2
%matplotlib inline

In [ ]:
from pathlib import Path
from torch.utils.data import WeightedRandomSampler
from fastai.callback.tracker import SaveModelCallback, Recorder
import matplotlib.pyplot as plt
import numpy as np
import cv2
import random
import torch
from mtrain.neg_mask.model.datasets.blur_pad_dl import random_tfm, BlurPadDataset, BlurPad8ChanDataset
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
import cv2
import random
from mtrain.utils import show, mkdir, DiskImage, DiskBooleanMask
from pytorch_grad_cam import (
    GradCAM,
)
import torch.nn as nn
import torch.nn.functional as F
from pytorch_grad_cam.utils.model_targets import ClassifierOutputTarget
from pytorch_grad_cam.utils.image import show_cam_on_image
from tqdm import tqdm
from mtrain.neg_mask.model.show import (
    get_preds_for_ds,
    show_classification_report,
    show_confusion_matrix,
    show_confusion_matrix_using_preds,
)
from mtrain.neg_mask.model.datasets.blur_pad_dl import CropTfmsOutsideBbox
from functools import partial
from sklearn.model_selection import train_test_split
from fastai.basics import DataLoaders, default_device
from mtrain.denorm import denormalize_imagenet, denormalize_4chan_imagenet
from mtrain.utils import show, it_chain
from fastai.callback.all import ProgressCallback
from fastai.basics import F1Score, Precision, Recall, CrossEntropyLossFlat
from fastai.vision.all import vision_learner, xresnet18

In [ ]:
from mtrain.utils import globL, mkdir

# CLEAN_PATH = Path(
#     "/Users/hariomnarang/Desktop/personal/roads/datasets/test-samples/neg-masking/V1/rocks/classification/blurred/clean/train"
# )
# FOVEATED_PATH = Path(
#     "/Users/hariomnarang/Desktop/personal/roads/datasets/test-samples/neg-masking/V1/rocks/classification/blurred/clean/foveated"
# )
FOVEATED_DS_PATH = Path(
    "/Users/hariomnarang/Desktop/personal/roads/datasets/test-samples/neg-masking/training/splits_dataset"
)
# UNIT_TEST_DEST = Path("/Users/hariomnarang/Desktop/personal/roads/datasets/unit-tests/negmask")
# mkdir(UNIT_TEST_DEST / "train")
# mkdir(UNIT_TEST_DEST / "masks")

# images = globL(CLEAN_PATH / "train", "*.jpg")

# import shutil
# images = images[:256]
# for img in images:
#     mask = CLEAN_PATH / "masks" / f"{img.stem}.png"

#     shutil.copy(img, UNIT_TEST_DEST / "train" / img.name)
#     shutil.copy(mask, UNIT_TEST_DEST / "masks" / mask.name)

DS_PATH = FOVEATED_DS_PATH
DS_PATH.exists()

In [ ]:
class SimpleCNN(nn.Module):
    def __init__(self, num_classes):
        super(SimpleCNN, self).__init__()
        
        self.encoder = nn.Sequential(
            nn.Conv2d(3, 16, kernel_size=3, padding=1, stride=2),
            nn.BatchNorm2d(16),
            nn.ReLU(),
            nn.Conv2d(16, 32, kernel_size=3, padding=1, stride=2),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.Conv2d(32, 64, kernel_size=3, padding=1, stride=2),
            nn.BatchNorm2d(64),
            nn.ReLU(),
        )
        # x = torch.flatten(x, 1) 
        # x = F.relu(self.fc1(x))
        # x = self.dropout(x)
        # x = self.fc2(x)
        self.head = nn.Sequential(
            nn.Flatten(),
            nn.Linear(64 * 8 * 8, 256),
            nn.ReLU(),
            nn.Dropout(0.25),
            nn.Linear(256, num_classes),
        )

    def forward(self, x):
        x = self.encoder(x)
        return self.head(x)

In [ ]:
def get_learner(dls, model=xresnet18):
    learn = vision_learner(
        dls,
        model,
        metrics=[F1Score(average="macro"), Precision(), Recall()],
        loss_func=CrossEntropyLossFlat(CLS_WEIGHT),
        n_out=2,
        normalize=False,
        pretrained=True,
    )
    learn.remove_cb(Recorder)
    learn.remove_cb(SaveModelCallback)
    learn.add_cb(Recorder())
    learn.add_cbs([SaveModelCallback(monitor="f1_score", fname="best")])
    learn = learn.remove_cb(ProgressCallback)
    return learn


def get_denormalized(tens):
    image, mask = None, None
    image = denormalize_imagenet(tens[:3])
    image = image.permute([1, 2, 0]).numpy()
    mask = tens[3]
    return image, mask


def show_gradcam_for_image(
    learn, input_tensor, target_label_idx=None, layer_name="0.7.1.conv1"
):
    target_layers = [learn.model.get_submodule(layer_name)]
    img_arr, _ = get_denormalized(input_tensor[0])

    targets = [ClassifierOutputTarget(target_label_idx)]

    with GradCAM(model=learn.model, target_layers=target_layers) as cam:
        grayscale_cam = cam(input_tensor=input_tensor, targets=targets)
        grayscale_cam = grayscale_cam[0, :]
        print(img_arr.shape, grayscale_cam.shape)
        visualization = show_cam_on_image(img_arr, grayscale_cam, use_rgb=True)
        model_outputs = cam.outputs

        return visualization, img_arr, model_outputs


def show_reports(learner):
    preds = learner.get_preds(dl=learner.dls.valid, with_decoded=True, with_loss=True)
    probs, targs, decoded, losses = preds
    labels = list(BlurPadDataset.LABEL_BY_IDX.keys())
    show_classification_report(probs, targs, labels)
    show_confusion_matrix_using_preds(probs, targs, labels)
    return probs, targs, decoded, losses

In [ ]:
from mtrain.neg_mask.model.datasets.blur_pad_dl import BlurPad4ChanDataset
CLS_WEIGHT = torch.tensor([1.0, 1.3]).float().to("mps")


def get_weight(p, taco_weight=1.0, manual_weight=1.2, default_weight=1.0):
    p = Path(p)
    if "taco" in p.name:
        return taco_weight
    if "manual" in p.name:
        return manual_weight
    else:
        return default_weight


def get_dls(
    num_samples,
    tfm=None,
    crop_size=224,
    ds_path=DS_PATH,
    min_area=35,
    min_bbox_length=3,
    max_area=None,
    path_filter=None,
    bs=8,
    bbox_pad=3,
):

    train_image_paths = list((ds_path / "train_ds" / "train").glob("*.jpg"))
    valid_image_paths = list((ds_path / "valid_ds" / "train").glob("*.jpg"))
    if path_filter is not None:
        train_image_paths = list(filter(path_filter, train_image_paths))
        valid_image_paths = list(filter(path_filter, valid_image_paths))
    train_image_paths = train_image_paths[:num_samples]
    valid_image_paths = valid_image_paths[: int(num_samples * 0.2)]

    train_ds = BlurPad8ChanDataset(
        train_image_paths,
        ds_path / "train_ds" / "masks",
        64,
        224,
        False,
        bbox_pad=bbox_pad,
        min_area=min_area,
        min_bbox_length=min_bbox_length,
        max_area=max_area,
    )
    valid_ds = BlurPad8ChanDataset(
        valid_image_paths,
        ds_path / "valid_ds" / "masks",
        64,
        224,
        True,
        bbox_pad=bbox_pad,
        min_area=min_area,
        min_bbox_length=min_bbox_length,
        max_area=max_area,
    )
    dls = DataLoaders.from_dsets(
        train_ds,
        valid_ds,
        device=default_device(),
        num_workers=4,
        bs=bs,
        persistent_workers=True,
    )
    return dls


def vis_sample_ds(ds, idx):
    sm, bg, label = ds[idx]
    img = denormalize_imagenet(bg.cpu()).permute([1,2,0])
    sm_img = denormalize_imagenet(sm.cpu()).permute([1,2,0])
    show([img, sm_img])


def vis_sample(dls, idx):
    vis_sample_ds(dls.train_ds, idx)

In [ ]:
# to counter the problem of the model focusing on texture/noise
# we decrease the probability of adding noise with each sweep while maintaining accuracy
# the next step is to remove overwrite noise
# then next is decreasing the add noise frequency
# first i would need to seee the performance of the model
#  on different types of aux transforms (step down? gaussian? blur?)
# our final model has no noise, and one kind of step down function
# we need to test it on all transforms and find the winner
# for each we do successive training by decreasing the add_noise chance parameter
def blur_tfm(
    cropped_image, mask, inner_bbox, add_noise_chance, blur_kernel_sz, blur_sigma
):
    add_noise = random.random() < add_noise_chance
    tfm = CropTfmsOutsideBbox(cropped_image, inner_bbox)
    tfm = tfm.overwrite_with_blur(blur_kernel_sz, blur_sigma)
    if add_noise:
        tfm = tfm.add_noise(20)
    return tfm.crop


def step_down_tfm(cropped_image, mask, inner_bbox, add_noise_chance, ratio):
    add_noise = random.random() < add_noise_chance
    tfm = CropTfmsOutsideBbox(cropped_image, inner_bbox)
    tfm = tfm.step_down(ratio)
    if add_noise:
        tfm = tfm.add_noise(20)
    return tfm.crop, mask


def step_down_gauss_tfm(cropped_image, mask, inner_bbox, add_noise_chance, min_value):
    add_noise = random.random() < add_noise_chance
    tfm = CropTfmsOutsideBbox(cropped_image, inner_bbox)
    tfm = tfm.step_down_gaussian(min_value)
    if add_noise:
        tfm = tfm.add_noise(20)
    return tfm.crop

In [ ]:
def get_initialised_learner():
    dls = get_dls(100, random_tfm)
    learner = get_learner(dls)
    MODELS_DIR = Path("/Users/hariomnarang/Desktop/personal/roads/datasets/models")
    path = MODELS_DIR / "foveated-224" / "iter-7-xresnet18.pth"
    state_dict = torch.load(path)
    learner.model.load_state_dict(state_dict)
    return learner

In [ ]:
st_ed_tfm = partial(step_down_tfm, ratio=0.5)

In [ ]:
def only_manual(path):
    stem = Path(path).stem
    is_wall_mapi = "mapillary" in stem or "taco" in stem
    return not is_wall_mapi


st_ed_tfm = partial(step_down_tfm, ratio=0.5)
st_ed_tfm0 = partial(st_ed_tfm, add_noise_chance=-1)
dls = get_dls(100, st_ed_tfm0, path_filter=only_manual, bbox_pad=5)

In [ ]:
vis_sample_ds(dls.train_ds, 3)

In [ ]:
import torch.nn as nn
from fastai.layers import AdaptiveConcatPool2d, Flatten
from torch.nn import functional as F

class CustomNOnlySm(nn.Module):
    def __init__(self, encoder1, head, *args, **kwargs):
        super().__init__()
        self.body1 = encoder1
        self.head = head

    def forward(self, sm_images, bg_images):
        acts1 = self.body1(sm_images)
        return self.head(acts1)

class CustomN(nn.Module):
    def __init__(self, encoder1, encoder2, head):
        super().__init__()
        self.body1, self.body2 = encoder1, encoder2
        self.head = head

        self.body1_pool_and_flatten = nn.Sequential(
            AdaptiveConcatPool2d(1),
            Flatten(full=False),
        )
        self.body2_pool_and_flatten = nn.Sequential(
            AdaptiveConcatPool2d(1),
            Flatten(full=False),
        )
        self.head = nn.Sequential(
            nn.BatchNorm1d(2048, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True),
            nn.Dropout(p=0.25, inplace=False),
            nn.Linear(in_features=2048, out_features=512, bias=False),
            nn.ReLU(inplace=True),
            nn.BatchNorm1d(512, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True),
            nn.Dropout(p=0.5, inplace=False),
            nn.Linear(in_features=512, out_features=2, bias=False),
        )
    def forward(self, sm_images, bg_images):
        acts1 = self.body1(sm_images)
        flattened_1 = self.body1_pool_and_flatten(acts1)
        acts2 = self.body2(bg_images)
        flattened_2 = self.body2_pool_and_flatten(acts2)
        flattened = torch.cat((flattened_1, flattened_2), dim=1)
        return self.head(flattened)

# First train the sm backbone

In [ ]:
from fastai.vision.all import params

def only_manual_and_taco(path):
    stem = Path(path).stem
    is_wall_mapi = "mapillary" in stem
    return not is_wall_mapi

def custom_n_sm_splitter(model):
    return [params(model.body1), params(model.head)]

In [ ]:
dls = get_dls(20_000, path_filter=only_manual)

In [ ]:
inner = SimpleCNN(2)
model = CustomNOnlySm(inner.encoder, inner.head)
learner = get_learner(dls)
learner.model = model

In [ ]:
learner.unfreeze()
learner.fit_one_cycle(10)

In [ ]:
learner.freeze()
learner.fit_one_cycle(2)

In [ ]:
learner.unfreeze()
learner.fit_one_cycle(15)

In [ ]:

baseline_learner.model[1]

In [ ]:
model = CustomNOnlySm(baseline_learner.model[0], baseline_learner.model[1])

dls = get_dls(100)
learner = get_learner(dls)
learner.model = model
learner.splitter = custom_n_sm_splitter
learner.dls.n_inp = 2

learner.fit_one_cycle(20)

In [ ]:
learner.model.mask_resize_to = 14

original_block = learner.model.body[7][0]

# 2. Modify the ConvPath: Change stride 2 -> 1
# Accessing: convpath -> ConvLayer(0) -> Conv2d
original_block.convpath[0][0].stride = (1, 1)

# 3. Modify the IdPath: Remove the AvgPool2d
# The original idpath is Sequential(AvgPool2d, ConvLayer)
# We replace it with just the ConvLayer to match the 14x14 spatial size
new_idpath = nn.Sequential(original_block.idpath[1]) 
original_block.idpath = new_idpath

In [ ]:
dls = get_dls(20000, st_ed_tfm0, path_filter=only_manual)

In [ ]:
learner.dls = dls

In [ ]:
learner.unfreeze()
learner.fit_one_cycle(2)

In [ ]:
learner.fit_one_cycle(4, 1e-3)

In [ ]:
learner.lr_find()

In [ ]:
learner.cbs

In [ ]:
learner.fit_one_cycle(10, lr_max=5e-4)

In [ ]:
torch.save(learner.model.state_dict(), "/Users/hariomnarang/Desktop/personal/roads/datasets/models/foveated-224/on-splits-last-act-14x14.pt")

In [ ]:
learner.fit_one_cycle

In [ ]:
def noise_overwriter(cropped_image, mask, inner_bbox):
    tfm = CropTfmsOutsideBbox(cropped_image, inner_bbox)
    tfm = tfm.overwrite_with_noise(40)
    return tfm.crop, mask


def train_from_scratch(learner, bs):
  path_filter = only_manual

  # first trin on fully noisy dataset
  # this trains the model to learn to look only at the interesting part of the image
  learner.unfreeze()

  print("STAGE: data=50, its=high, tfm=noise_overwriter")

  dls = get_dls(50, noise_overwriter, path_filter=path_filter, bs=bs)
  learner.dls = dls
  learner.fit_one_cycle(20)


  print("STAGE: data=1000, its=high, tfm=noise_overwriter")
  dls = get_dls(1000, noise_overwriter, path_filter=path_filter, bs=bs)
  learner.dls = dls
  learner.fit_one_cycle(10)

  print("STAGE: data=ALL, its=high, tfm=noise_overwriter")
  dls = get_dls(20000, noise_overwriter, path_filter=path_filter, bs=bs)
  learner.dls = dls
  learner.fit_one_cycle(4)

  print("STAGE: data=100, its=high, tfm=random_tfm")
  dls = get_dls(50, random_tfm, path_filter=path_filter, bs=bs)
  learner.dls = dls
  learner.fit_one_cycle(20)

  print("STAGE: data=500, its=high, tfm=random_tfm")
  dls = get_dls(500, random_tfm, path_filter=path_filter, bs=bs)
  learner.dls = dls
  learner.fit_one_cycle(10)

  print("STAGE: data=1000, its=high, tfm=random_tfm")
  dls = get_dls(1000, random_tfm, path_filter=path_filter, bs=bs)
  learner.dls = dls
  learner.fit_one_cycle(5)

  print("STAGE: data=5000, its=high, tfm=random_tfm")
  dls = get_dls(5000, random_tfm, path_filter=path_filter, bs=bs)
  learner.dls = dls
  learner.fit_one_cycle(2)

  print("STAGE: data=20000, its=high, tfm=random_tfm")
  dls = get_dls(20000, random_tfm, path_filter=path_filter, bs=bs)
  learner.dls = dls
  learner.fit_one_cycle(2)


  st_ed_tfm_50 = partial(st_ed_tfm, add_noise_chance=0.5)
  st_ed_tfm_15 = partial(st_ed_tfm, add_noise_chance=0.15)
  st_ed_tfm_0 = partial(st_ed_tfm, add_noise_chance=-1)

  print("STAGE: data=20000, its=high, tfm=step-down-50")
  dls = get_dls(20000, st_ed_tfm_50, path_filter=path_filter, bs=bs)
  learner.dls = dls
  learner.fit_one_cycle(2)

  print("STAGE: data=20000, its=high, tfm=step-down-15")
  dls = get_dls(20000, st_ed_tfm_15, path_filter=path_filter, bs=bs)
  learner.dls = dls
  learner.fit_one_cycle(2)

  print("STAGE: data=20000, its=high, tfm=step-down-0")
  dls = get_dls(20000, st_ed_tfm_0, path_filter=path_filter, bs=bs)
  learner.dls = dls
  learner.fit_one_cycle(2)

  print("STAGE: data=20000, its=high, tfm=step-down-0 unfrozen")
  # learner.unfreeze()
  learner.fit_one_cycle(2)

  return learner
  # we would again like to train this on some randomised version of the dataset

In [ ]:
learner = get_new_learner(get_dls(500, noise_overwriter))

In [ ]:
trained_learner = train_from_scratch(learner, 8)

In [ ]:
with torch.no_grad():
    trained_learner.model.eval()
    _ = show_reports(trained_learner)

Two unfrozen cycles:
```
STAGE: data=20000, its=high, tfm=step-down-0 unfrozen
[0, 0.3182360529899597, 0.4363847076892853, 0.7634135322244942, 0.842809364548495, 0.5185185185185185, '02:29']
[1, 0.22781486809253693, 0.3623899519443512, 0.8209795574307523, 0.724007561436673, 0.7880658436213992, '02:31']
```

In [ ]:
trained_learner.train()
trained_learner.lr_find()

In [ ]:
trained_learner.add_cb(SaveModelCallback())

In [ ]:
trained_learner.fit_one_cycle(4, 1e-5)

In [ ]:
with torch.no_grad():
    trained_learner.model.eval()
    res = show_reports(trained_learner)
    probs, targs, decoded, losses = res
    sorted_losses = list(reversed(sorted([(loss, i) for i, loss in enumerate(losses)])))
    top_loss_idxs = [sl[1] for sl in sorted_losses]
    min_losses = [sl[1] for sl in reversed(sorted_losses)]

In [ ]:
lrn = trained_learner
vds = lrn.dls.valid_ds
i = top_loss_idxs[10]
print("target", targs[i], "decoded", decoded[i])
viz, img, mo = show_gradcam_for_image(lrn, vds[i][0].unsqueeze(0), 1, "body.7.1.convpath.1.0")
show([viz, img])

In [ ]:
torch.save(lrn.model.state_dict(), "/Users/hariomnarang/Desktop/personal/roads/datasets/models/foveated-224/on-splits.pt")

In [ ]:
class EmbeddingCollector:
    def __init__(self, model, layer_idx):
        # We hook into layer (1): the Flatten layer
        # This gives us the 1024-dim vector (Concat of Avg & Max pool)
        self.target_layer = model.get_submodule(layer_idx)
        self.hook = self.target_layer.register_forward_hook(self.hook_fn)
        self.stored = None

    def hook_fn(self, m, i, o):
        # o is the output of the Flatten layer
        self.stored = o.detach().cpu()

    def remove(self): self.hook.remove()

In [ ]:
# def precompute_all(learn, valid_ds, layer_idx=1):
learn = lrn
collector = EmbeddingCollector(learn.model, "body.7.1.convpath.1.0")
embs = []

learn.model.eval()
test_dl = learn.dls.test_dl(learn.dls.valid_ds, with_labels=True, shuffle=False, bs=1)
with torch.no_grad():
    for b in tqdm(test_dl):
        _ = learn.model(b[0])
        embs.append(collector.stored)

collector.remove()
# return torch.cat(embs)

In [ ]:
# i want to see the 8th layers outputs, inputs and the layer itself
last_weight = next(learn.model.head[8].parameters())
last_weight.shape
# this shape 2 x 512

# each 512 weight is applied to the inputs
# 0th weight is for other
# 1st weight is for trash

In [ ]:
weight = last_weight.detach().clone().to("cpu").numpy()

In [ ]:
plt.plot(weight[0])
plt.plot(weight[1], color="red")

This does not give us any indication of what the weight is doing lol.  
Which input values were the most dominant in the final sum?

You find the piece-wise mults of them. Then plot a hist.  
Each value can be assigned a ratio of how much it is affecting the final output (simply the sum val / total abs sum) (or sum? i want the relative magnitude, so the total abs sum is gud).  


given an input activation (a vector of shape [512]), an output activation (a vector of shape [2]) and weight ([2, 512]).   

The output activation defines how much it affects the final sum. Binary classification is the simplest, higher values point to class 1, lower to class 0.  
Each output activation has some value, which divided by its full abs sum is the contribution of that layer.  

from the contribution of that output activation, we know the corresponding input activation. The contribution of this activation in the final output is the contribution of its output activation.  
Either the linear layer weight is strong in this, or the input activation is.   

We have some value `10 = 2*5`. The contribution of `2` and `5` is?

for `2+5 = 7`, 2's contribution is `2/7`, `5` is `5/7`.  
for `2-5=-3` it is `2/-3` and `-5/-3`? No, it should be `2/7` and `-5/7` (-3/7 is the output which is fine ratio wise).  

What about multiplication?   


In [ ]:
embs[0].shape